In [1]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


In [2]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.fc1   = nn.Linear(16 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, 10)
        self.pool  = nn.MaxPool2d(2)
        self.relu  = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

In [3]:
model = TinyNet()
model.load_state_dict(torch.load('tinynet_baseline.pth'))
model.eval()
print("Baseline model loaded!")




Baseline model loaded!


In [5]:
import os
print(os.getcwd())

c:\Users\hafsa\Desktop\TinyML_code


In [4]:
import os
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.pth'):
            print(os.path.join(root, file))

.\tinynet_baseline.pth


In [6]:
def check_sparsity(model):
    total = 0
    zeros = 0
    for name, param in model.named_parameters():
        if 'weight' in name:
            total+= param.nelement()
            zeros += float(torch.sum(param ==0))
    print(f"Global sparsity: {100 * zeros / total:.1f}%")
check_sparsity(model)

Global sparsity: 0.0%


# PRUNING

In [7]:
prune.l1_unstructured(model.conv1, name='weight', amount=0.4)
prune.l1_unstructured(model.conv2, name='weight', amount=0.4)
prune.l1_unstructured(model.fc1, name='weight', amount=0.4)
prune.l1_unstructured(model.fc2, name='weight', amount=0.4)

check_sparsity(model)



Global sparsity: 0.0%


# SAME SPARSITY BECAUSE OF MASKING

In [8]:
def check_sparsity(model):
    total = 0
    zeros = 0
    for name, module in model.named_modules():
        if isinstance(module,(nn.Conv2d,nn.Linear) ):
            weight = module.weight
            total+= weight.nelement()
            zeros += float(torch.sum(weight ==0))
    print(f"Global sparsity: {100 * zeros / total:.1f}%")
check_sparsity(model)

Global sparsity: 40.0%


Did pruning hurt the accuracy? By how much?

In [9]:
# Define test loader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_data = torchvision.datasets.MNIST('./data', train=False, 
                                        transform=transform, download=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Evaluate function
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            output = model(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Accuracy: {100 * correct / total:.2f}%")

evaluate(model, test_loader)

Accuracy: 98.56%


Testing Sparsity levels

In [10]:
def load_fresh_model():
    model = TinyNet()
    model.load_state_dict(torch.load('tinynet_baseline.pth'))
    model.eval()

In [11]:
def prune_model(model,amount):
    prune.l1_unstructured(model.conv1, name='weight', amount=amount)
    prune.l1_unstructured(model.conv2, name='weight', amount=amount)
    prune.l1_unstructured(model.fc1, name='weight', amount=amount)
    prune.l1_unstructured(model.fc2, name='weight', amount=amount)
    return model
